# ESE 561 Take-Home Midterm: From Game-Tree Search to Monte Carlo Tree Search

**20 points (+1 bonus).** Work alone. You may use the lecture notes, the textbook and the Python/NumPy documentation, and AI. Submit this notebook **executed**, with every written answer in the Markdown cells marked *Your answer*.

**Oral check.** After submission each of you will have a 5-10-minute presentation in class: the questions will be assigned beforehand.

Monte Carlo tree search (MCTS) is the bridge between the first half of the course and the second (you might find helpful [this video](https://youtu.be/UXW2yZndl7U?si=hrsVWGkTkcBvHp6y)):

* it is **adversarial search** (minimax) in which exhaustive expansion is replaced by **sampling**;
* every node of the tree is a **multi-armed bandit**, and the choice of child uses the **UCB** (upper confidence bound) rule from the lecture;
* AlphaGo and AlphaZero are MCTS plus neural networks trained by **reinforcement learning**.

Minimax on tic-tac-toe and UCB were built in class; they are provided or recalled where needed. You will run the implementation of alpha-beta (Part 1) and MCTS (Part 2) on tic-tac-toe and Connect Four, compare them (Part 3) and answer theory questions (Part 4). Each part starts with the theory it uses.

| Part | Topic | Points |
|---|---|---|
| 1 | Minimax, alpha-beta, depth limits | 4 |
| 2 | Monte Carlo tree search | 3 |
| 3 | Experiments and analysis | 4 |
| 4 | Theory, and AlphaGo/AlphaZero | 6 |
| Bonus | Play | 1 |
| 5 | In-class presentation | 3 |

**Running time.** Everything runs on a laptop CPU; the longest cell takes about a minute.

In [1]:
import math, random, time
import numpy as np
import matplotlib.pyplot as plt
random.seed(0); np.random.seed(0)

## 0. The games (provided, do not modify)

**Tic-tac-toe.** Two players, `X` (moves first) and `O`, take turns marking an empty square of a $3\times3$ grid. The first to get three of their marks in a row (horizontally, vertically or diagonally) wins; if all nine squares are filled without that, the game is a draw. A move is a square number:
```
0 1 2
3 4 5
6 7 8
```
With perfect play by both sides the game is a **draw**. It is small (549,946 nodes in the full game tree), so it can be solved exactly by minimax: we use it to test our algorithms against perfect play.

**Connect Four.** A vertical board of 7 columns and 6 rows. The players (`X` first, then `O`) take turns dropping a disc into a column; the disc falls to the **lowest empty cell** of that column, so a move is just a column number 0–6 (at most 7 legal moves). The first player to get **four in a row** horizontally, vertically or diagonally wins; if all 42 cells are filled without that, it is a draw.
```
. . . . . . .      row 5 (top)
. . . . . . .
. . . . . . .
. . . . . . .
. . . O . . .
. X X X O . O      row 0 (bottom)
0 1 2 3 4 5 6      columns = moves
```
Connect Four has about $4.5\cdot10^{12}$ reachable positions. It was solved in 1988 (J. Allen; V. Allis): the **first player wins** with perfect play, starting in the center column. Central cells belong to more lines of four than edge cells, which is why center moves are usually best. It is far too large for minimax in Python, which is why we need depth limits, evaluation functions and MCTS.

**Rewards.** There is **no reward during the game**, only at the end: the utility of a terminal state is
$$U(s,p)=\begin{cases}+1 & p \text{ won},\\ \ \ 0 & \text{draw},\\ -1 & p \text{ lost},\end{cases}$$
and $U(s,1-p)=-U(s,p)$ (zero-sum). There is no discount ($\gamma=1$): the value of a position is the outcome of the game under the assumed play, so a win in 3 moves and a win in 13 moves are worth the same $+1$. Three different numbers appear in the algorithms; do not confuse them:

| where | what | range |
|---|---|---|
| minimax, alpha-beta at a terminal state | the utility $U$ | $\{-1,0,+1\}$ |
| alpha-beta at the depth limit | the **estimate** `c4_eval`, not a reward | $(-1,1)$ |
| MCTS backup | the same utility rescaled to $(U+1)/2$, so that $W/N$ is a **win rate** (draw = ½) | $\{0,\tfrac12,1\}$ |

The rescaling in MCTS is needed because UCB1 assumes rewards in $[0,1]$: its exploration constant $c=\sqrt2$ is calibrated for that range (Hoeffding's inequality).

Both games share one interface; every algorithm you write uses only these methods, so it works for both games unchanged:

| Method | Returns |
|---|---|
| `initial()` | the start state |
| `to_move(state)` | the player to move: 0 (moves first, `X`) or 1 (`O`) |
| `moves(state)` | list of legal moves (empty if the game is over) |
| `result(state, move)` | the next state (states are immutable) |
| `terminal(state)` | is the game over? |
| `utility(state, player)` | at a terminal state: +1 if `player` won, −1 if lost, 0 for a draw |
| `show(state)` | prints the board |

In [2]:
class TicTacToe:
    """state = (board, n): board is a tuple of 9 cells in {0: empty, 1: X, 2: O}; n = moves played."""
    LINES = [(0,1,2),(3,4,5),(6,7,8),(0,3,6),(1,4,7),(2,5,8),(0,4,8),(2,4,6)]
    name = 'tic-tac-toe'

    def initial(self):
        return ((0,) * 9, 0)

    def to_move(self, state):
        return state[1] % 2

    def moves(self, state):
        if self.terminal(state):
            return []
        return [i for i, c in enumerate(state[0]) if c == 0]

    def result(self, state, move):
        board, n = state
        b = list(board); b[move] = n % 2 + 1
        return (tuple(b), n + 1)

    def winner(self, state):
        b = state[0]
        for i, j, k in self.LINES:
            if b[i] and b[i] == b[j] == b[k]:
                return b[i] - 1
        return None

    def terminal(self, state):
        return self.winner(state) is not None or state[1] == 9

    def utility(self, state, player):
        w = self.winner(state)
        return 0 if w is None else (1 if w == player else -1)

    def show(self, state):
        sym = '.XO'
        print('\n'.join(' '.join(sym[state[0][3*r + c]] for c in range(3)) for r in range(3)))


class ConnectFour:
    """7 columns x 6 rows, stored as bitboards (J. Tromp's representation).
    state = (cur, mask, n, won): cur = stones of the player to move, mask = all stones,
    n = moves played, won = the player who just moved has four in a row.
    Cell (column c, row r from the bottom) is bit 7*c + r; a move is a column 0..6."""
    name = 'Connect Four'

    def initial(self):
        return (0, 0, 0, False)

    def to_move(self, state):
        return state[2] % 2

    def moves(self, state):
        cur, mask, n, won = state
        if won or n == 42:
            return []
        # center columns first: a good move ordering for alpha-beta
        return [c for c in (3, 2, 4, 1, 5, 0, 6) if not mask & (1 << (7 * c + 5))]

    @staticmethod
    def _aligned(pos):
        for s in (1, 7, 6, 8):                       # vertical, horizontal, two diagonals
            m = pos & (pos >> s)
            if m & (m >> 2 * s):
                return True
        return False

    def result(self, state, move):
        cur, mask, n, won = state
        new_mask = mask | (mask + (1 << (7 * move)))  # drop a stone into column `move`
        mover = cur | (new_mask ^ mask)               # stones of the player who just moved
        return (cur ^ mask, new_mask, n + 1, self._aligned(mover))

    def terminal(self, state):
        return state[3] or state[2] == 42

    def winner(self, state):
        return (state[2] - 1) % 2 if state[3] else None

    def utility(self, state, player):
        w = self.winner(state)
        return 0 if w is None else (1 if w == player else -1)

    def stones(self, state, player):
        cur, mask, n, _ = state
        return cur if n % 2 == player else cur ^ mask

    def show(self, state):
        p0, p1 = self.stones(state, 0), self.stones(state, 1)
        rows = [' '.join('X' if p0 >> (7*c + r) & 1 else 'O' if p1 >> (7*c + r) & 1 else '.'
                         for c in range(7)) for r in range(5, -1, -1)]
        print('\n'.join(rows) + '\n0 1 2 3 4 5 6')


TTT, C4 = TicTacToe(), ConnectFour()
s = C4.initial()
for m in [3, 3, 2, 4, 1, 6]:
    s = C4.result(s, m)
C4.show(s)
print('player to move:', 'X' if C4.to_move(s) == 0 else 'O')
print('X to move wins immediately with column(s):', [m for m in C4.moves(s) if C4.terminal(C4.result(s, m))])

. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . O . . .
. X X X O . O
0 1 2 3 4 5 6
player to move: X
X to move wins immediately with column(s): [0]


**Agents and matches (provided).** An *agent* is a function `agent(game, state) -> move`. `match` plays `n_games` games, alternating who moves first, and returns `(A wins, draws, B wins)`.

For Connect Four we also provide a hand-made **evaluation function** `c4_eval(game, state, player)` in $[-1,1]$: it counts the windows of four cells that contain only one player's stones (more stones = more weight).

In [3]:
def random_agent(game, state):
    return random.choice(game.moves(state))


def play(game, agents, verbose=False):
    """agents[0] moves first. Returns the winner (0 or 1) or None for a draw."""
    s = game.initial()
    while not game.terminal(s):
        s = game.result(s, agents[game.to_move(s)](game, s))
    if verbose:
        game.show(s)
    return game.winner(s)


def match(game, agent_a, agent_b, n_games):
    a = d = b = 0
    for g in range(n_games):
        if g % 2 == 0:
            w = play(game, [agent_a, agent_b]); a += w == 0; b += w == 1
        else:
            w = play(game, [agent_b, agent_a]); a += w == 1; b += w == 0
        d += w is None
    return a, d, b


C4_WINDOWS = []
for c in range(7):
    for r in range(6):
        for dc, dr in ((1, 0), (0, 1), (1, 1), (1, -1)):
            cells = [(c + k*dc, r + k*dr) for k in range(4)]
            if all(0 <= cc < 7 and 0 <= rr < 6 for cc, rr in cells):
                C4_WINDOWS.append(sum(1 << (7*cc + rr) for cc, rr in cells))
WEIGHT = {0: 0, 1: 1, 2: 4, 3: 16}

def c4_eval(game, state, player):
    me, op = game.stones(state, player), game.stones(state, 1 - player)
    score = 0
    for w in C4_WINDOWS:
        a, b = w & me, w & op
        if a and not b:
            score += WEIGHT[bin(a).count('1')]
        elif b and not a:
            score -= WEIGHT[bin(b).count('1')]
    return math.tanh(score / 50)

print(len(C4_WINDOWS), 'windows of four')

69 windows of four


## Part 1. Minimax, alpha-beta and depth limits (4 pt)

### 1.1 Games as search problems

Our games are **two-player, zero-sum, deterministic, with perfect information**: both players see the whole board, nothing is random, and one player's gain is the other's loss. Such a game is a **game tree**: nodes are states, edges are moves, and the levels alternate between the two players. Leaves are terminal states with utility $U(s,p)\in\{-1,0,+1\}$ for player $p$.

**Minimax value.** If both players play optimally, the value of a state for player $p$ is defined backwards from the leaves (backward induction, Zermelo 1913):
$$
V(s)=\begin{cases}
U(s,p), & s \text{ terminal},\\[2pt]
\max_{a} V(\text{result}(s,a)), & p \text{ to move (MAX)},\\[2pt]
\min_{a} V(\text{result}(s,a)), & \text{the opponent to move (MIN)}.
\end{cases}
$$
MAX picks the move with the largest value: the best move **against an opponent who answers every move with its best reply**. Compare with the Bellman optimality equation of the RL part of the course: the same recursion, with the expectation over the environment replaced by a minimum over the opponent's moves.

**Cost.** Minimax is a depth-first search of the whole tree: time $O(b^d)$, memory $O(bd)$, for branching factor $b$ and depth $d$. Tic-tac-toe ($b\le9$, $d\le9$): 549,946 nodes, about one second in Python. Connect Four ($b=7$, $d\le42$): about $4.5\cdot10^{12}$ positions. Chess ($b\approx35$, $d\approx80$): about $10^{123}$ nodes.

The minimax function from class is provided below, together with a **depth-limited** version that stops after `depth` moves and returns a heuristic estimate `eval_fn` there (Section 1.3).

In [4]:
def minimax(game, state, player, counter):
    """Minimax value of `state` for `player` (from class). counter[0] counts the nodes visited."""
    counter[0] += 1
    if game.terminal(state):
        return game.utility(state, player)
    vals = [minimax(game, game.result(state, m), player, counter) for m in game.moves(state)]
    return max(vals) if game.to_move(state) == player else min(vals)


def minimax_depth(game, state, player, depth, eval_fn, counter):
    """The same, but stops after `depth` moves and returns eval_fn there."""
    counter[0] += 1
    if game.terminal(state):
        return game.utility(state, player)
    if depth == 0:
        return eval_fn(game, state, player)
    vals = [minimax_depth(game, game.result(state, m), player, depth - 1, eval_fn, counter) for m in game.moves(state)]
    return max(vals) if game.to_move(state) == player else min(vals)


cnt = [0]; t = time.time()
v = minimax(TTT, TTT.initial(), 0, cnt)
print(f'value of tic-tac-toe for X: {v};  nodes visited: {cnt[0]:,};  {time.time() - t:.1f} s')

value of tic-tac-toe for X: 0;  nodes visited: 549,946;  2.6 s


### 1.2 Alpha-beta pruning: the same value, far fewer nodes

**Why prune?** Minimax looks at every node, but many of them cannot change the decision. Suppose MAX has already found a move worth $3$. While evaluating a second move, MAX finds that MIN has a reply to it worth $2$. Then the second move is worth **at most** $2<3$ for MAX, whatever MIN's other replies are, and MAX will never play it: the remaining replies need not be searched. *A move is refuted as soon as one sufficiently good answer to it is found.*

```
            MAX
          /     \
       MIN=3    MIN<=2         after the reply worth 2, the other
      / | \     / |  \         replies to the second move are pruned
     3 12  8   2  x   x
```

**The two bounds.** Alpha-beta passes two numbers down the recursion:
* $\alpha$: the value that MAX can already **guarantee** on the current path (a lower bound, starts at $-\infty$);
* $\beta$: the value that MIN can already guarantee (an upper bound, starts at $+\infty$).

Only values strictly between $\alpha$ and $\beta$ can still affect the decision at the root. At a MAX node, as soon as its value reaches $v\ge\beta$, return: MIN, one level up, already has a better option and will never let the game reach this node (a **beta cut-off**). Symmetrically, at a MIN node return as soon as $v\le\alpha$ (an **alpha cut-off**). Otherwise update $\alpha\leftarrow\max(\alpha,v)$ at MAX nodes and $\beta\leftarrow\min(\beta,v)$ at MIN nodes.

**Correctness.** If the true minimax value lies in $(\alpha,\beta)$, alpha-beta returns it exactly; otherwise it returns a bound on the correct side, which is all the parent needs. Called at the root with $(-\infty,+\infty)$ it returns **exactly** the minimax value: pruning loses nothing.

**Move ordering.** How much is pruned depends on the order of the moves. If the best move is always searched first, alpha-beta examines about $b^{d/2}$ leaves instead of $b^d$ (Knuth and Moore, 1975; you prove this in Q4.1); with random order about $b^{3d/4}$; with the worst move first, nothing is pruned. This is why `ConnectFour.moves` lists the center columns first: central moves are usually the best ones.

**Why we need it.** In the same time, $b^{d/2}$ instead of $b^d$ means searching about **twice as deep**, and in depth-limited search (Section 1.3) depth is playing strength. It also makes our experiments feasible: the perfect tic-tac-toe player below runs a full search before every move, and Part 3 plays dozens of Connect Four games against a depth-4 searcher.

### Q1.1 (0 pt). Alpha-beta with a depth limit.
Implement `alphabeta(game, state, player, depth, alpha, beta, eval_fn, counter)`:
* at a terminal state return the utility;
* when `depth == 0` return `eval_fn(game, state, player)` (a heuristic estimate, Section 1.3);
* otherwise search the children with alpha-beta pruning, with `depth - 1`;
* increase `counter[0]` at every call, if a counter is given.

With `depth = math.inf` it must return the exact minimax value. Check that on tic-tac-toe.

In [5]:
def alphabeta(game, state, player, depth, alpha=-math.inf, beta=math.inf, eval_fn=None, counter=None):
    if counter is not None:
        counter[0] += 1
    if game.terminal(state):
        return game.utility(state, player)
    if depth == 0:
        return eval_fn(game, state, player)
    if game.to_move(state) == player:
        v = -math.inf
        for m in game.moves(state):
            v = max(v, alphabeta(game, game.result(state, m), player, depth - 1, alpha, beta, eval_fn, counter))
            if v >= beta:
                return v
            alpha = max(alpha, v)
        return v
    v = math.inf
    for m in game.moves(state):
        v = min(v, alphabeta(game, game.result(state, m), player, depth - 1, alpha, beta, eval_fn, counter))
        if v <= alpha:
            return v
        beta = min(beta, v)
    return v


cnt = [0]
v = alphabeta(TTT, TTT.initial(), 0, math.inf, counter=cnt)
print(f'alpha-beta value: {v};  nodes visited: {cnt[0]:,}')

alpha-beta value: 0;  nodes visited: 18,297


The agent below (provided) uses your `alphabeta`: it searches `depth` moves ahead and breaks ties at random, so repeated games differ. `make_alphabeta_agent(9)` plays tic-tac-toe perfectly.

In [6]:
def make_alphabeta_agent(depth, eval_fn=None):
    def agent(game, state):
        p = game.to_move(state)
        best, best_moves = -math.inf, []
        for m in game.moves(state):
            v = alphabeta(game, game.result(state, m), p, depth - 1, eval_fn=eval_fn)
            if v > best + 1e-12:
                best, best_moves = v, [m]
            elif abs(v - best) <= 1e-12:
                best_moves.append(m)
        return random.choice(best_moves)
    return agent

perfect_ttt = make_alphabeta_agent(9)
print('perfect vs random, tic-tac-toe (perfect wins, draws, random wins):',
      match(TTT, perfect_ttt, random_agent, 20))

perfect vs random, tic-tac-toe (perfect wins, draws, random wins): (15, 5, 0)


### Watching alpha-beta (provided)

`alphabeta_trace` is the same algorithm as your `alphabeta`, but it prints every node: the move that leads to it, whether it is a MAX or MIN node, the window $(\alpha,\beta)$ it receives, the values that come back, and every **cut-off** with the moves it prunes. Values are from the point of view of the player to move at the root (MAX).

The example is a tic-tac-toe position with four empty squares, `O` to move. O wins at once with square 5 (row 3–4–5); square 2 blocks X's threat on the top row (0–1–2) and leads to a draw; squares 7 and 8 let X win. Watch how, once square 5 is known to be worth +1, the other moves are refuted after a single X reply.

`explain_alphabeta` shows how the agent **chooses** a move: the value of every legal move (each searched with a full window, so all values are exact), and the best one.

In [7]:
def fmt(v):
    return '-inf' if v == -math.inf else '+inf' if v == math.inf else f'{v:+.3g}'

def alphabeta_trace(game, state, player, depth, alpha=-math.inf, beta=math.inf, eval_fn=None, move=None, indent=0):
    """alphabeta that prints the search tree. Returns the same value as alphabeta."""
    pad = '|   ' * indent
    kind = 'MAX' if game.to_move(state) == player else 'MIN'
    head = f'{pad}{"root" if move is None else "move " + str(move)}'
    if game.terminal(state):
        u = game.utility(state, player)
        print(f'{head}: terminal, utility {u:+d}'); return u
    if depth == 0:
        e = eval_fn(game, state, player)
        print(f'{head}: depth limit, eval {e:+.3f}'); return e
    print(f'{head} [{kind}, to move: {"XO"[game.to_move(state)]}]  window ({fmt(alpha)}, {fmt(beta)})')
    moves = game.moves(state)
    v = -math.inf if kind == 'MAX' else math.inf
    for i, m in enumerate(moves):
        child = alphabeta_trace(game, game.result(state, m), player, depth - 1, alpha, beta, eval_fn, m, indent + 1)
        if kind == 'MAX':
            v = max(v, child)
            if v >= beta:
                print(f'{pad}  beta cut-off: v = {fmt(v)} >= beta = {fmt(beta)}; pruned moves {moves[i + 1:]} -> return {fmt(v)}')
                return v
            alpha = max(alpha, v)
        else:
            v = min(v, child)
            if v <= alpha:
                print(f'{pad}  alpha cut-off: v = {fmt(v)} <= alpha = {fmt(alpha)}; pruned moves {moves[i + 1:]} -> return {fmt(v)}')
                return v
            beta = min(beta, v)
    print(f'{pad}  -> {kind} value {fmt(v)}')
    return v


def explain_alphabeta(game, state, depth, eval_fn=None):
    """How make_alphabeta_agent chooses: the value of every move for the player to move."""
    p = game.to_move(state)
    vals, cnt, t = {}, [0], time.time()
    for m in game.moves(state):
        vals[m] = alphabeta(game, game.result(state, m), p, depth - 1, eval_fn=eval_fn, counter=cnt)
    best = max(vals.values())
    choice = random.choice([m for m, v in vals.items() if abs(v - best) <= 1e-12])
    print(f'alpha-beta depth {depth}, {"XO"[p]} to move: ' +
          ', '.join(f'{m}: {v:+.3f}' for m, v in vals.items()) +
          f'   -> plays {choice}   ({cnt[0]:,} nodes, {time.time() - t:.2f} s)')
    return choice


s = TTT.initial()
for m in [0, 3, 1, 4, 6]:                                   # X: 0, 1, 6   O: 3, 4
    s = TTT.result(s, m)
TTT.show(s); print()
v = alphabeta_trace(TTT, s, TTT.to_move(s), math.inf)
print('\nvalue for the player to move:', v, '\n')
explain_alphabeta(TTT, s, 9)

X X .
O O .
X . .

root [MAX, to move: O]  window (-inf, +inf)
|   move 2 [MIN, to move: X]  window (-inf, +inf)
|   |   move 5 [MAX, to move: O]  window (-inf, +inf)
|   |   |   move 7 [MIN, to move: X]  window (-inf, +inf)
|   |   |   |   move 8: terminal, utility +0
|   |   |     -> MIN value +0
|   |   |   move 8 [MIN, to move: X]  window (+0, +inf)
|   |   |   |   move 7: terminal, utility +0
|   |   |     alpha cut-off: v = +0 <= alpha = +0; pruned moves [] -> return +0
|   |     -> MAX value +0
|   |   move 7 [MAX, to move: O]  window (-inf, +0)
|   |   |   move 5: terminal, utility +1
|   |     beta cut-off: v = +1 >= beta = +0; pruned moves [8] -> return +1
|   |   move 8 [MAX, to move: O]  window (-inf, +0)
|   |   |   move 5: terminal, utility +1
|   |     beta cut-off: v = +1 >= beta = +0; pruned moves [7] -> return +1
|     -> MIN value +0
|   move 5: terminal, utility +1
|   move 7 [MIN, to move: X]  window (+1, +inf)
|   |   move 2: terminal, utility -1
|     alpha cut-o

5

### Q1.2 (3 pt). How much does pruning buy?
Count the nodes visited (and measure the time) for
* tic-tac-toe, full search: `minimax` vs `alphabeta`;
* Connect Four from the empty board with `c4_eval` at the horizon, for `depth = 2, 4, 6`: `minimax_depth` vs `alphabeta`, and `alphabeta` on the game `C4_LEFT_TO_RIGHT` defined below, which is the same game with the moves listed in the order 0, 1, …, 6 instead of center-first. Also run `alphabeta` with center-first ordering at depth 8.

Then answer:
1. Do all searches return the same value? By what factor does alpha-beta reduce the work on tic-tac-toe?
2. Roughly what factor per **extra ply** (per added level of depth) do you observe for minimax, for alpha-beta with center-first ordering and for alpha-beta with left-to-right ordering? Compare with $b=7$ and $\sqrt7\approx2.6$.
3. How deep can alpha-beta search in the time minimax needs for depth 6? Why does move ordering matter so much?

In [8]:
class ConnectFourLeftToRight(ConnectFour):
    """The same game, but the moves are listed left to right: a poor ordering for alpha-beta."""
    def moves(self, state):
        return sorted(super().moves(state))

C4_LEFT_TO_RIGHT = ConnectFourLeftToRight()

In [9]:
print('tic-tac-toe, full search')
for name, f in [('minimax', lambda c: minimax(TTT, TTT.initial(), 0, c)),
                ('alpha-beta', lambda c: alphabeta(TTT, TTT.initial(), 0, math.inf, counter=c))]:
    c = [0]; t = time.time(); v = f(c)
    print(f'  {name:10s} value {v}   {c[0]:9,} nodes   {time.time() - t:.2f} s')

print('\nConnect Four, empty board, c4_eval at the horizon (nodes; value)')
print(f'{"depth":>5}  {"minimax":>17}  {"alpha-beta":>17}  {"a-b left to right":>17}')
for d in [2, 4, 6, 8]:
    row = []
    for g, f in [(C4, minimax_depth if d <= 6 else None), (C4, alphabeta), (C4_LEFT_TO_RIGHT, alphabeta if d <= 6 else None)]:
        if f is None:
            row.append(f'{"-":>17}'); continue
        c = [0]
        v = (f(g, g.initial(), 0, d, c4_eval, c) if f is minimax_depth
             else f(g, g.initial(), 0, d, eval_fn=c4_eval, counter=c))
        row.append(f'{c[0]:>9,}; {v:+.4f}')
    print(f'{d:>5}  ' + '  '.join(row))

tic-tac-toe, full search
  minimax    value 0     549,946 nodes   2.63 s
  alpha-beta value 0      18,297 nodes   0.09 s

Connect Four, empty board, c4_eval at the horizon (nodes; value)
depth            minimax         alpha-beta  a-b left to right
    2         57; -0.0599         24; -0.0599         42; -0.0599
    4      2,801; -0.0400        189; -0.0400        858; -0.0400
    6    137,257; -0.0400      1,407; -0.0400     13,224; -0.0400
    8                  -     19,909; -0.0798                  -


*Your answer:*

### 1.3 Depth limits and evaluation functions

Even with perfect ordering, $b^{d/2}$ is far too much for Connect Four or chess to the end of the game. Shannon (1950) proposed to **cut the search** at a fixed depth and replace the true value of the positions there by a heuristic **evaluation function** $\text{Eval}(s,p)$:
$$
V_d(s)=\begin{cases}U(s,p), & s\text{ terminal},\\ \text{Eval}(s,p), & d=0,\\ \max/\min_a V_{d-1}(\text{result}(s,a)) & \text{otherwise.}\end{cases}
$$
A good evaluation function (i) agrees with the utility at terminal states, (ii) is ordered like the chance of winning, (iii) is cheap. Ours, `c4_eval`, counts windows of four cells that only one player can still complete, with more weight for windows that are nearly full: it encodes **human knowledge** about Connect Four. Chess programs used material (pawn = 1, knight = 3, …) plus positional terms.

**What it costs.** The result is no longer the minimax value: it is the minimax value of a different game that ends after $d$ moves and is scored by Eval. The search is blind beyond its horizon. A threat that matures one move after the horizon is invisible, and a search may even play delaying moves that push an unavoidable loss beyond its horizon (the **horizon effect**). Remedies are searching "noisy" positions deeper (quiescence search) and better evaluation functions, i.e. more knowledge.

### Q1.3 (1 pt). Why depth limits and evaluation functions?
1. Explain why `alphabeta(C4, C4.initial(), 0, math.inf)` is hopeless in Python even with pruning (use your measurements from Q1.2 and $d\le42$), and what a depth limit and an evaluation function trade away.
2. Describe a concrete Connect Four situation in which depth-4 alpha-beta with `c4_eval` plays a losing move that a deeper search would avoid.

*Your answer:*

## Part 2. Monte Carlo tree search (3 pt)

### 2.1 Evaluating a position by sampling

Alpha-beta needs an evaluation function, i.e. knowledge. MCTS replaces it by **simulation**: to estimate how good a position is, play many games from it with random moves to the end and average the results. This is **Monte Carlo evaluation**, exactly as in Monte Carlo prediction in the RL part of the course: the value is estimated by the average of sampled returns.

* It needs no knowledge beyond the rules: only a simulator (`result`, `terminal`, `utility`).
* It estimates the value under **random** play, not under best play. Still, a position in which one side has more ways to win under random play is usually also better under good play.

**Flat Monte Carlo** gives every legal move the same number $n$ of random playouts and picks the move with the best average. It has two defects:
1. it wastes playouts on clearly bad moves;
2. it averages over the opponent's replies instead of taking the minimum: a move with one crushing refutation and many harmless replies looks good.

### 2.2 From flat Monte Carlo to a tree: UCT

MCTS fixes both defects by growing a **tree** of the positions that matter and choosing where to sample by solving a **bandit problem at every node**: each move of a node is an arm, its reward is the result of a playout through it. With UCB1 from the lecture,
$$
\text{child}=\arg\max_{\text{child}}\Big[\frac{W_{\text{child}}}{N_{\text{child}}}+c\sqrt{\frac{\ln N_{\text{node}}}{N_{\text{child}}}}\Big],
$$
the first term (exploitation) is the average result of the playouts through the child, and the second (exploration) is large for children that were tried rarely. Moves that look good get most playouts (fixing defect 1), and in the subtree of a move the **opponent's** good replies get most playouts, so the average of that move is dominated by its best refutation (fixing defect 2, gradually). This is **UCT**, "UCB applied to trees" (Kocsis and Szepesvári, 2006). They proved that, as the number of playouts grows, the probability of choosing a wrong move at the root goes to zero: UCT converges to the minimax move (Q4.2).

In the language of the course, MCTS is **generalized policy iteration** inside the tree: the playouts **evaluate** the moves (Monte Carlo averages $W/N$, estimates of $Q(s,a)$), and UCT **improves** the policy that chooses moves by acting nearly greedily on these estimates, with an exploration bonus instead of ε.

Properties that matter in practice:
* **anytime**: stop after any number of playouts and play the best move found so far;
* **asymmetric**: the tree grows deep along promising lines and stays shallow elsewhere, unlike the uniform depth of alpha-beta;
* **knowledge-free**: but knowledge can be added (a better rollout policy in the bonus; neural networks in AlphaZero, Part 4).

### 2.3 The algorithm
MCTS grows a search tree one node at a time. Each **playout** has four steps:

1. **Selection**: from the root, while the current node is fully expanded (no untried moves) and has children, move to the child chosen by **UCT** = UCB1 at that node:
   $\displaystyle \arg\max_{\text{child}}\Big[\frac{W_{\text{child}}}{N_{\text{child}}}+c\sqrt{\frac{\ln N_{\text{node}}}{N_{\text{child}}}}\Big]$;
2. **Expansion**: if the node has untried moves, pick one at random, create its child and move to it;
3. **Simulation (rollout)**: from the new node, play random moves until the game ends;
4. **Backup**: walk back to the root; at every node add 1 to $N$ and add the reward of the final state **for the player who made the move into that node**: 1 for a win, 0.5 for a draw, 0 for a loss.

After all playouts, play the root's **most visited** child.

Storing the reward from the point of view of "the player who moved into the node" makes selection correct for both players: at every node the player to move picks the child that is best for **themselves**.

Why the most visited child and not the one with the highest average? The average of a rarely visited child is unreliable, and UCT visits a child often only if its average stayed high.

In [ ]:
class Node:
    def __init__(self, game, state, parent=None, move=None):
        self.state, self.parent, self.move = state, parent, move
        self.children = []                         # expanded children
        self.untried = list(game.moves(state))     # moves not expanded yet
        self.N, self.W = 0, 0.0                    # visits, total reward
        self.player = 1 - game.to_move(state)      # the player who made the move INTO this node

### Q2.1 (0 pt). UCT selection.
Return the child of `node` that maximizes the UCT score. (All children have `N >= 1` when this is called.)

In [ ]:
def uct_select(node, c=math.sqrt(2)):
    log_n = math.log(node.N)
    return max(node.children, key=lambda ch: ch.W / ch.N + c * math.sqrt(log_n / ch.N))

### Q2.2 (0 pt). Random rollout.
Play uniformly random moves from `state` until the game ends; return the final state.

In [ ]:
def rollout(game, state):
    while not game.terminal(state):
        state = game.result(state, random.choice(game.moves(state)))
    return state

### Q2.3 (0 pt). The MCTS loop.
Implement `mcts(game, root_state, n_playouts, c)` following the four steps above. Return the most visited move **and** the root (we inspect the tree later). The parameter `rollout_fn` lets you swap in a different rollout policy in Q4.4.

In [ ]:
def mcts(game, root_state, n_playouts, c=math.sqrt(2), rollout_fn=None):
    rollout_fn = rollout_fn or rollout
    root = Node(game, root_state)
    for _ in range(n_playouts):
        node = root
        # 1. selection
        while not node.untried and node.children:
            node = uct_select(node, c)
        # 2. expansion
        if node.untried:
            m = node.untried.pop(random.randrange(len(node.untried)))
            child = Node(game, game.result(node.state, m), parent=node, move=m)
            node.children.append(child)
            node = child
        # 3. simulation
        final = rollout_fn(game, node.state)
        # 4. backup
        while node is not None:
            node.N += 1
            node.W += (game.utility(final, node.player) + 1) / 2
            node = node.parent
    best = max(root.children, key=lambda ch: ch.N)
    return best.move, root


def make_mcts_agent(n_playouts, c=math.sqrt(2), rollout_fn=None):
    def agent(game, state):
        return mcts(game, state, n_playouts, c, rollout_fn)[0]
    return agent

**Tests.** A correct MCTS should (a) find an immediate win, (b) block an immediate threat, (c) almost never lose to a random player at tic-tac-toe with 200 playouts, and (d) draw more and more often against perfect play as the number of playouts grows.

In [ ]:
# (a) X has 1, 2, 3 in the bottom row (O holds column 4): X to move wins with column 0
s = C4.initial()
for m in [3, 3, 2, 4, 1, 6]:
    s = C4.result(s, m)
print('(a) MCTS move:', mcts(C4, s, 2000)[0], '(expected 0)')

# (b) X has 0, 1, 2 in the bottom row: O to move must block column 3
s = C4.initial()
for m in [0, 6, 1, 6, 2]:
    s = C4.result(s, m)
print('(b) MCTS move:', mcts(C4, s, 2000)[0], '(expected 3)')

# (c) and (d)
print('(c) MCTS(200) vs random:', match(TTT, make_mcts_agent(200), random_agent, 20), ' (MCTS wins, draws, random wins)')
for n in [50, 200, 1000, 5000]:
    print(f'(d) MCTS({n}) vs perfect:', match(TTT, make_mcts_agent(n), perfect_ttt, 20))

### Watching MCTS (provided)

`mcts_trace` is the same algorithm as your `mcts`, but for the first few playouts it prints each of the four steps:
* **selection**: at every fully expanded node, the UCT score of every child, $W/N$ + bonus, and the child chosen;
* **expansion**: the new move added to the tree;
* **rollout**: the random moves played to the end, and the outcome;
* **backup**: the reward added to every node on the path, from the point of view of the player who moved into that node (1 win, ½ draw, 0 loss).

At the end it prints the root statistics: the visits $N$ and win rate $W/N$ of every move, and the move chosen (most visited).

The position: tic-tac-toe, X has two opposite corners (0 and 8), O has the center, **O to move**. The only correct replies are the **edges** 1, 3, 5, 7: a corner (2 or 6) loses, because X takes the other corner and has two threats at once (a fork). The first six playouts only expand the six root moves; selection by UCT starts at playout 7. Watch where the visits go.

In [ ]:
def mcts_trace(game, root_state, n_playouts, show=4, c=math.sqrt(2), rollout_fn=None):
    rollout_fn = rollout_fn or rollout
    root = Node(game, root_state)
    name = lambda node: 'root' if node.parent is None else '-'.join(str(n.move) for n in path_to(node))
    for i in range(n_playouts):
        v = i < show
        if v: print(f'=== playout {i + 1}')
        node = root
        while not node.untried and node.children:                         # 1. selection
            scores = {ch.move: ch.W / ch.N + c * math.sqrt(math.log(node.N) / ch.N) for ch in node.children}
            node = uct_select(node, c)
            if v: print(f'  select at {name(node.parent)} (N={node.parent.N}): UCT ' +
                        ', '.join(f'{m}: {s_:.2f}' for m, s_ in scores.items()) + f'  -> {node.move}')
        if node.untried:                                                   # 2. expansion
            m = node.untried.pop(random.randrange(len(node.untried)))
            child = Node(game, game.result(node.state, m), parent=node, move=m)
            node.children.append(child); node = child
            if v: print(f'  expand move {m} (new node {name(node)})')
        s, moves = node.state, []                                          # 3. rollout (recorded)
        if rollout_fn is rollout:
            while not game.terminal(s):
                mv = random.choice(game.moves(s)); moves.append(mv); s = game.result(s, mv)
            final = s
        else:
            final = rollout_fn(game, s)
        w = game.winner(final)
        if v: print(f'  rollout {moves if moves else "(none: terminal)"} -> ' +
                    ('draw' if w is None else f'{"XO"[w]} wins'))
        rewards = []
        while node is not None:                                            # 4. backup
            r = (game.utility(final, node.player) + 1) / 2
            node.N += 1; node.W += r
            rewards.append(f'{name(node)}[{"XO"[node.player]}] +{r:g} -> W/N={node.W:g}/{node.N}')
            node = node.parent
        if v: print('  backup: ' + ';  '.join(rewards))
    print(f'\nafter {n_playouts} playouts:  move: N, W/N')
    for ch in sorted(root.children, key=lambda ch: -ch.N):
        print(f'  {ch.move}: {ch.N:5d}, {ch.W / ch.N:.3f}')
    best = max(root.children, key=lambda ch: ch.N)
    print('plays', best.move)
    return best.move, root

def path_to(node):
    p = []
    while node.parent is not None:
        p.append(node); node = node.parent
    return p[::-1]


def explain_mcts(game, state, n_playouts, rollout_fn=None):
    """How make_mcts_agent chooses: visits and win rate of every move at the root."""
    t = time.time()
    move, root = mcts(game, state, n_playouts, rollout_fn=rollout_fn)
    print(f'MCTS {n_playouts} playouts, {"XO"[game.to_move(state)]} to move:  move N (W/N): ' +
          ', '.join(f'{ch.move} {ch.N} ({ch.W / ch.N:.2f})' for ch in sorted(root.children, key=lambda ch: ch.move)) +
          f'   -> plays {move}   ({time.time() - t:.2f} s)')
    return move


random.seed(1)
s = TTT.initial()
for m in [0, 4, 8]:                                      # X: 0, 8   O: 4;  O to move
    s = TTT.result(s, m)
TTT.show(s); print()
mcts_trace(TTT, s, 3000, show=8);

### Q2.4 (3 pt). Explain the tests.
1. In test (d), why does MCTS with few playouts lose to perfect play, even though tic-tac-toe is tiny?
2. Inspect the tree after 5000 playouts from the empty tic-tac-toe board: print `N` and `W/N` for each root child. Which first moves does MCTS prefer, and does this match what you know about the game?
3. What does `W/N` of a root child estimate? Why is it **not** the minimax value of that move?

In [ ]:
_, root = mcts(TTT, TTT.initial(), 5000)
for ch in sorted(root.children, key=lambda ch: -ch.N):
    print(f'square {ch.move}: N = {ch.N:5d}, W/N = {ch.W / ch.N:.3f}')


*Your answer:*

## Part 3. Experiments and analysis (4 pt)

### 3.1 Knowledge versus search

The two players we compare get their strength from different places.

| | alpha-beta, depth $d$ | MCTS |
|---|---|---|
| which moves are searched | all of them, to depth $d$ (pruning removes only irrelevant ones) | selected by UCT: promising lines deep, others shallow |
| how a position is evaluated | `c4_eval` at the horizon: **human knowledge** | random playouts to the end of the game: **no knowledge**, but noisy |
| what improves it | depth (time) and a better Eval | playouts (time) and better rollouts |

**Computer Go.** In Go ($19\times19$, $b\approx250$, $d\approx150$) alpha-beta failed for decades: the branching factor kills depth, and no good evaluation function was known, because the value of a position depends on territory that is settled only at the end, and counting stones means little in the middle of the game. Programs played at weak amateur level. In 2006–2008 MCTS programs (Crazy Stone, MoGo) needed no evaluation function at all, and jumped to strong amateur level, first on $9\times9$ and then on $19\times19$. AlphaGo (2016) added learned networks to MCTS (Part 4).

### Q3.1 (2 pt). MCTS vs. alpha-beta on Connect Four.
Play 10 games (alternating who starts) for each combination below, and plot MCTS's score (wins + ½ draws, as a fraction) against the number of playouts:

* MCTS with 100, 400 and 1600 playouts per move,
* against depth-4 alpha-beta **with** `c4_eval`, and against depth-4 alpha-beta **without** an evaluation function (`eval_fn = lambda g, s, p: 0.0`, so it only sees wins and losses within 4 moves).

Then answer:
1. Which opponent is harder for MCTS, and why? What plays the role of the evaluation function inside MCTS?
2. MCTS uses no knowledge of Connect Four beyond the rules. Alpha-beta with `c4_eval` uses human knowledge. Relate your results to why MCTS made computer Go possible, where no good evaluation function was known.

In [ ]:
budgets = [100, 400, 1600]
zero_eval = lambda g, s, p: 0.0
opponents = {'alpha-beta d4 + c4_eval': make_alphabeta_agent(4, c4_eval),
             'alpha-beta d4, no evaluation': make_alphabeta_agent(4, zero_eval)}
scores = {}
t0 = time.time()
for name, opp in opponents.items():
    scores[name] = []
    for n in budgets:
        w, d, l = match(C4, make_mcts_agent(n), opp, 10)
        scores[name].append((w + d / 2) / 10)
        print(f'MCTS({n:4d}) vs {name}: {w} wins, {d} draws, {l} losses   ({time.time() - t0:.0f} s)')
for name, sc in scores.items():
    plt.plot(budgets, sc, 'o-', label=f'vs {name}')
plt.xscale('log'); plt.ylim(-.05, 1.05); plt.axhline(.5, c='gray', ls='--')
plt.xlabel('MCTS playouts per move'); plt.ylabel('MCTS score'); plt.legend(); plt.grid(alpha=.3); plt.show()

*Your answer:*

### Q3.2 (1 pt). Where does MCTS spend its playouts?
Run 3000 playouts from the empty Connect Four board and plot the visit count `N` of each root child (column 0–6) as a bar chart. Which column does MCTS consider best? How does UCT decide to "stop exploring" a column?

In [ ]:
_, root = mcts(C4, C4.initial(), 3000)
visits = {ch.move: ch.N for ch in root.children}
plt.bar(range(7), [visits[c] for c in range(7)])
plt.xlabel('column'); plt.ylabel('visits N'); plt.title('Root visits after 3000 playouts'); plt.show()
print({c: round(ch.W / ch.N, 3) for c, ch in sorted((ch.move, ch) for ch in root.children)})

*Your answer:*

### Q3.3 (1 pt). When MCTS is the wrong tool.
In the lecture notebook on the 4×3 grid world, UCT with random playouts reached an average return of only about 0.4 with 160 playouts per move, while value iteration gives the exact optimum (0.7453) in milliseconds. Give two reasons why a small MDP with a known model favors dynamic programming over MCTS, and two properties of Go or Connect Four that favor MCTS.

*Your answer:*

## Part 4. Theory, and AlphaGo/AlphaZero (6 pt)

### 4.1 From MCTS to AlphaGo and AlphaZero: what the networks learn, and from which data

Your experiments show the two weaknesses of plain MCTS: random rollouts are a **noisy, biased evaluation** (they estimate the value under random play), and UCT **spreads playouts over all moves**, including obviously bad ones, before it can focus. DeepMind's programs fix both with neural networks; the important question is what data trains them.

**AlphaGo** (Silver et al., 2016; beat Fan Hui and Lee Sedol). Four networks, trained one after another:

| network | role in the search | trained on | target |
|---|---|---|---|
| SL policy $p_\sigma(a\mid s)$, 13-layer CNN | prior $P(s,a)$ that biases selection towards plausible moves | about 30 million positions from about 160,000 games of strong human amateurs (KGS server) | the move the human played (classification; 57% accuracy) |
| fast rollout policy $p_\pi$, linear softmax on local patterns | plays the rollouts (about 1000× faster than $p_\sigma$) | human games | the human move (about 24% accuracy) |
| RL policy $p_\rho$, initialized from $p_\sigma$ | not used in the search; generates data for $v_\theta$ | **self-play** games against randomly chosen earlier versions of itself | policy gradient (REINFORCE) with reward $\pm1$ at the end of the game; wins about 80% against $p_\sigma$ |
| value network $v_\theta(s)$ | evaluates the leaf, mixed half and half with a rollout result | 30 million positions, **each from a different self-play game** of $p_\rho$ | the outcome $z=\pm1$ of that game (regression) |

Why one position per game: consecutive positions of a game differ by one stone and share the same label $z$; trained on whole games, the network memorized the outcomes of the games instead of learning to evaluate positions (it overfitted).

So AlphaGo learns its prior and rollouts from **humans**, and its evaluation from **its own games**.

**AlphaGo Zero** (2017) and **AlphaZero** (2017–18, also chess and shogi) remove the humans and the rollouts. One residual network $f_\theta(s)=(p,v)$ with a policy head and a value head starts from **random weights** and is trained only on its own games:
1. **Self-play with search.** At every move of every game, run MCTS guided by the current network (800 simulations per move in AlphaZero): the bonus is $c\,P(s,a)\frac{\sqrt{N(s)}}{1+N(s,a)}$ with $P=p_\theta$, and a new leaf is evaluated by $v_\theta$ instead of a rollout. Play a move drawn from the visit distribution $\pi(a\mid s)\propto N(s,a)^{1/\tau}$.
2. **Data.** Every position $s_t$ of every game is stored with **two labels**: the search probabilities $\pi_t$ (the root visit counts of the MCTS run at $s_t$) and the final outcome $z_t\in\{-1,0,+1\}$ of the game from the point of view of the player to move at $s_t$.
3. **Training.** Mini-batches of recent positions (a replay buffer, as in DQN) minimize
$$\ell(\theta)=(z-v_\theta(s))^2-\pi^\top\log p_\theta(s)+c\|\theta\|^2.$$

The scale: AlphaGo Zero's first run played 4.9 million games in three days; AlphaZero trained for 700,000 steps of 4096 positions on about 44 million games of chess (24 million of shogi).

**In the language of the course.** The visit distribution $\pi$ is better than the raw network policy $p_\theta$, because the search has looked ahead: **MCTS is a policy improvement operator**. Training $p_\theta$ towards $\pi$ copies the improved policy into the network; training $v_\theta$ towards the outcome $z$ is **Monte Carlo policy evaluation** of the improved policy (the target is a sampled return, not a TD target). Self-play, search and training together are **generalized policy iteration**, with a network that generalizes across positions instead of a table. The next tree search starts from a better prior and a better evaluation: this is how the results of one search are reused in all later games.

**Q4.1 (1 pt).** With perfect move ordering alpha-beta examines $O(b^{d/2})$ leaves instead of $b^d$ for a uniform tree of branching factor $b$ and depth $d$. What does this mean for how deep alpha-beta can search in the same time as minimax? *(A careful argument for $d=2$ and $d=4$ is enough, with the general pattern stated.)*

**Q4.1 (1 pt).** Our UCT backs up the **average** reward. Minimax backs up the **max/min**. Explain why averaging is the right choice early in the search, when a node has few visits, and why UCT still converges to the minimax move as the number of playouts grows (an intuitive argument is enough).

**Q4.2 (3 pt).** AlphaZero changes MCTS in two places: the rollout is replaced by the **value head** $v_\theta(s)$, and the UCT bonus becomes $c\,P_\theta(s,a)\frac{\sqrt{N(s)}}{1+N(s,a)}$ with the **policy head** $P_\theta$.
1. For each change, say which weakness of plain MCTS it addresses, using your results from Part 3.
2. The policy head is trained on the MCTS visit distribution $\pi$, not on the moves of human experts (as in AlphaGo) and not on its own output. Why is $\pi$ a good target, and what would go wrong if the network were trained to imitate its own raw policy $p_\theta$?
3. The value head is trained on the final outcome $z$ of self-play games. Which method from the RL part of the course does this correspond to, and why does AlphaGo use only one position per game for this regression?

*Your answers:*

### Q4.4 (1 pt). A smarter rollout.
Random rollouts miss obvious moves. Write `smart_rollout(game, state)` that, at every step of the rollout, (1) plays a move that wins immediately if there is one, else (2) plays a random move among those after which the opponent **cannot** win immediately (in Connect Four this blocks threats), else (3) plays at random. Compare MCTS(400) with random rollouts and with smart rollouts against depth-4 alpha-beta with `c4_eval` (10 games each). Is a smarter but slower rollout worth it at a fixed **time** budget? Measure the time per move.

In [ ]:
def smart_rollout(game, state):
    while not game.terminal(state):
        me = game.to_move(state)
        nxt = [game.result(state, m) for m in game.moves(state)]
        win = [s2 for s2 in nxt if game.winner(s2) == me]
        if win:
            state = win[0]
            continue
        safe = [s2 for s2 in nxt
                if not any(game.winner(game.result(s2, m2)) == 1 - me for m2 in game.moves(s2))]
        state = random.choice(safe or nxt)
    return state


ab4 = make_alphabeta_agent(4, c4_eval)
for name, fn in [('random rollout', None), ('smart rollout', smart_rollout)]:
    t = time.time()
    r = match(C4, make_mcts_agent(400, rollout_fn=fn), ab4, 10)
    print(f'MCTS(400), {name}: {r}   {time.time() - t:.0f} s for 10 games')

*Your answer:*

## Bonus. Play against the machine (1 pt)

Play tic-tac-toe or Connect Four against the algorithms of this notebook and compare them yourself. Before every machine move the notebook prints **why** it plays that move:
* `random`: no reasoning;
* `alphabeta`: the value of every legal move (+1 forced win, −1 forced loss, in between: `c4_eval` at the depth limit);
* `mcts`: the visits $N$ and win rate $W/N$ of every move; it plays the most visited;
* `mcts_smart`: MCTS with the smart rollout of the bonus.

Enter a square (tic-tac-toe) or a column (Connect Four); `q` quits. Things to try: does depth-2 alpha-beta fall for a double threat? How many playouts does MCTS need before you cannot beat it? In which positions do alpha-beta and MCTS disagree?

The interactive cell is switched off so that "Run all" does not wait for input: set `INTERACTIVE = True` to play. The cell after it lets two machines play each other with the same explanations.

In [ ]:
def make_explaining_agent(kind, depth=4, playouts=1000, eval_fn=None):
    """An agent that prints how it chooses each move."""
    def agent(game, state):
        if kind == 'random':
            m = random_agent(game, state); print(f'random plays {m}'); return m
        if kind == 'alphabeta':
            return explain_alphabeta(game, state, depth, eval_fn if game is C4 else None)
        if kind == 'mcts':
            return explain_mcts(game, state, playouts)
        if kind == 'mcts_smart':
            return explain_mcts(game, state, playouts, rollout_fn=smart_rollout)
        raise ValueError(kind)
    return agent


def human_agent(game, state):
    legal = game.moves(state)
    while True:
        a = input(f'your move {legal} (q to quit): ').strip()
        if a == 'q':
            raise KeyboardInterrupt
        if a.isdigit() and int(a) in legal:
            return int(a)
        print('not a legal move')


def play_vs_machine(game, machine='mcts', human_first=True, depth=4, playouts=1000, eval_fn=c4_eval):
    agents = [human_agent, make_explaining_agent(machine, depth, playouts, eval_fn)]
    if not human_first:
        agents.reverse()
    s = game.initial()
    try:
        while not game.terminal(s):
            print(); game.show(s)
            print(f'{"XO"[game.to_move(s)]} to move')
            s = game.result(s, agents[game.to_move(s)](game, s))
    except KeyboardInterrupt:
        print('game abandoned'); return
    print(); game.show(s)
    w = game.winner(s)
    print('draw' if w is None else ('you win!' if agents[w] is human_agent else 'the machine wins'))


def watch(game, kind_x, kind_o, **kw):
    """Two explaining agents play each other."""
    agents = [make_explaining_agent(kind_x, **kw), make_explaining_agent(kind_o, **kw)]
    s = game.initial()
    while not game.terminal(s):
        print(f'{"XO"[game.to_move(s)]}:', end=' ')
        s = game.result(s, agents[game.to_move(s)](game, s))
    game.show(s)
    w = game.winner(s)
    print('draw' if w is None else f'{"XO"[w]} ({[kind_x, kind_o][w]}) wins')


INTERACTIVE = False
if INTERACTIVE:
    play_vs_machine(C4, machine='alphabeta', human_first=True, depth=4)
    # other opponents:
    # play_vs_machine(C4, machine='mcts', playouts=2000)
    # play_vs_machine(C4, machine='mcts_smart', playouts=400, human_first=False)
    # play_vs_machine(TTT, machine='alphabeta', depth=9)          # perfect play: you can only draw

In [ ]:
random.seed(3)
watch(C4, 'alphabeta', 'mcts', depth=4, playouts=1000, eval_fn=c4_eval)